In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('spam.csv', encoding='latin-1')

# Latin-1 encoding is used to read the CSV file, which may contain special characters.

# Exploratory Data Analysis

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [6]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [7]:
# checking for missing values
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [8]:
# checking for duplicates for each column
for col in df.columns:
    print(f"{col} : {df[col].duplicated().sum()}")

v1 : 5570
v2 : 403
Unnamed: 2 : 5528
Unnamed: 3 : 5561
Unnamed: 4 : 5566


# Handling missing values

In this case , we see that there are no missing values for the v1 and v2 columns. the missing values are mainly in the Unnamed : 2 , Unnamed : 3 , Unnamed : 4 columns , which has mostly null values. so dropping them.

In [9]:
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
df.columns

Index(['v1', 'v2'], dtype='object')

# Handling duplicate values

In [10]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

**Convert the labels to 0 and 1 :**
* 0: not spam (ham)
* 1: spam

In [11]:
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

**Separate the labels**

In [12]:
X = df['v2']
y = df['v1']

**Split dataset into training and testing data**

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=35)

# TF_IDF Vectorization for the words in the v2 column

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf  = TfidfVectorizer(stop_words='english' , ngram_range=(1,2) , max_features=5000 , sublinear_tf=True)
X_train_new = tfidf.fit_transform(X_train)
X_test_new = tfidf.transform(X_test)

# Creating a dictionary for storing the metrics for various different models

In [42]:
accuracy_dict = {}
precision_dict = {}
recall_dict = {}
f1_dict = {}

# Model : Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression

# default params used first 

lr = LogisticRegression()
lr.fit(X_train_new, y_train)

LogisticRegression()

In [16]:
y_pred_lr = lr.predict(X_test_new)

**check accuracy**

In [43]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score 

print("Accuracy: ", accuracy_score(y_test, y_pred_lr))
print("Precision: ", precision_score(y_test, y_pred_lr))
print("Recall: ", recall_score(y_test, y_pred_lr))  
print("F1 Score: ", f1_score(y_test, y_pred_lr))

accuracy_dict['Logistic Regression'] = accuracy_score(y_test, y_pred_lr)
precision_dict['Logistic Regression'] = precision_score(y_test, y_pred_lr)
recall_dict['Logistic Regression'] = recall_score(y_test, y_pred_lr)
f1_dict['Logistic Regression'] = f1_score(y_test, y_pred_lr)

Accuracy:  0.9698375870069605
Precision:  0.9921259842519685
Recall:  0.7682926829268293
F1 Score:  0.865979381443299


# Model : Logistic Regression with hyperparameter tuning 

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

lr_ht = LogisticRegression(max_iter=800,random_state=42,solver='liblinear')

# add penalty and class_weight to improve the accuracy :) 
pgrid = {'C' : [0.01,0.1,1] , 'penalty':['l1','l2'] , 'class_weight':[None,'balanced']}

grid_log_reg = GridSearchCV(lr_ht,pgrid,cv=3,scoring='f1',n_jobs=-1)
grid_log_reg.fit(X_train_new,y_train)

GridSearchCV(cv=3,
             estimator=LogisticRegression(max_iter=800, random_state=42,
                                          solver='liblinear'),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1],
                         'class_weight': [None, 'balanced'],
                         'penalty': ['l1', 'l2']},
             scoring='f1')

In [19]:
grid_log_reg.best_estimator_

LogisticRegression(C=1, class_weight='balanced', max_iter=800, random_state=42,
                   solver='liblinear')

In [20]:
y_pred_lr_ht = grid_log_reg.best_estimator_.predict(X_test_new)

In [44]:
print("Accuracy: ", accuracy_score(y_test, y_pred_lr_ht))
print("Precision: ", precision_score(y_test, y_pred_lr_ht))
print("Recall: ", recall_score(y_test, y_pred_lr_ht))  
print("F1 Score: ", f1_score(y_test, y_pred_lr_ht))

accuracy_dict['Logistic Regression with hyper parameter tuning'] = accuracy_score(y_test, y_pred_lr_ht)
precision_dict['Logistic Regression with hyper parameter tuning'] = precision_score(y_test, y_pred_lr_ht)
recall_dict['Logistic Regression with hyper parameter tuning'] = recall_score(y_test, y_pred_lr_ht)
f1_dict['Logistic Regression with hyper parameter tuning'] = f1_score(y_test, y_pred_lr_ht)

Accuracy:  0.9783449342614076
Precision:  0.9047619047619048
Recall:  0.926829268292683
F1 Score:  0.9156626506024096


# Model : XGBoost 

In [22]:
%pip install xgboost

import xgboost as xgb

xgb_clf = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
xgb_clf.fit(X_train_new, y_train)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Arunima\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [23]:
y_pred_xgb = xgb_clf.predict(X_test_new)

**check accuracy**

In [45]:
print("Accuracy: ", accuracy_score(y_test, y_pred_xgb))
print("Precision: ", precision_score(y_test, y_pred_xgb))
print("Recall: ", recall_score(y_test, y_pred_xgb))
print("F1 Score: ", f1_score(y_test, y_pred_xgb))

accuracy_dict['XGBoost Classifier'] = accuracy_score(y_test, y_pred_xgb)
precision_dict['XGBoost Classifier'] = precision_score(y_test, y_pred_xgb)
recall_dict['XGBoost Classifier'] = recall_score(y_test, y_pred_xgb)
f1_dict['XGBoost Classifier'] = f1_score(y_test, y_pred_xgb)

Accuracy:  0.9752513534416086
Precision:  0.9459459459459459
Recall:  0.8536585365853658
F1 Score:  0.8974358974358975


**poor _Recall_ and _F1_ scores**

# Model : Hyper parameter tuning on the XGBoost model

In [29]:
from sklearn.model_selection import GridSearchCV

params_xgb = {'n_estimators': [200,500,700],
            'learning_rate' : [0.01, 0.03, 0.05, 0.1, 0.2],
              'max_depth' : [3,5,7]
        }

grid_xgb = GridSearchCV(xgb.XGBClassifier(random_state=42, n_jobs=-1), params_xgb , cv=3, scoring='f1' , n_jobs = -1)
grid_xgb.fit(X_train_new, y_train)

GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [200, 500, 700]},
             scoring='f1')

In [30]:
grid_xgb.best_estimator_

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=-1, num_parallel_tree=None, ...)

In [31]:
y_pred_xgb_ht = grid_xgb.best_estimator_.predict(X_test_new)

**check accuracy**

In [46]:
print("Accuracy: ", accuracy_score(y_test, y_pred_xgb_ht))
print("Precision: ", precision_score(y_test, y_pred_xgb_ht))
print("Recall: ", recall_score(y_test, y_pred_xgb_ht))
print("F1 Score: ", f1_score(y_test, y_pred_xgb_ht))

accuracy_dict['XGBoost Classifier with hyper parameter tuning'] = accuracy_score(y_test, y_pred_xgb_ht)
precision_dict['XGBoost Classifier with hyper parameter tuning'] = precision_score(y_test, y_pred_xgb_ht)
recall_dict['XGBoost Classifier with hyper parameter tuning'] = recall_score(y_test, y_pred_xgb_ht)
f1_dict['XGBoost Classifier with hyper parameter tuning'] = f1_score(y_test, y_pred_xgb_ht) 

Accuracy:  0.974477958236659
Precision:  0.9395973154362416
Recall:  0.8536585365853658
F1 Score:  0.8945686900958466


this signifies that the model without hyper parameter tuning as well was giving optimal results.

# Model : Multinomial Naive Bayes

In [35]:
from sklearn.naive_bayes import MultinomialNB
 
# default params first
mnb = MultinomialNB()
mnb.fit(X_train_new, y_train)

MultinomialNB()

In [36]:
y_pred_mnb = mnb.predict(X_test_new)

**check accuracy**

In [47]:
print("Accuracy: ", accuracy_score(y_test, y_pred_mnb))
print("Precision: ", precision_score(y_test, y_pred_mnb))
print("Recall: ", recall_score(y_test, y_pred_mnb))
print("F1 Score: ", f1_score(y_test, y_pred_mnb))

accuracy_dict['Multinomial Naive Bayes'] = accuracy_score(y_test, y_pred_mnb)
precision_dict['Multinomial Naive Bayes'] = precision_score(y_test, y_pred_mnb)
recall_dict['Multinomial Naive Bayes'] = recall_score(y_test, y_pred_mnb)
f1_dict['Multinomial Naive Bayes'] = f1_score(y_test, y_pred_mnb)

Accuracy:  0.974477958236659
Precision:  1.0
Recall:  0.7987804878048781
F1 Score:  0.888135593220339


High precision and accuracy ! 

But , low Recall and F1 score!

# Model : Hyper parameter tuning on Multinomial Naive Bayes

In [38]:
from sklearn.model_selection import GridSearchCV

params_mnb = {'alpha': [0.1, 0.5, 1.0, 2.0, 5.0]}

grid_mnb = GridSearchCV(MultinomialNB(), params_mnb, cv=5, scoring='f1' , n_jobs=-1)
grid_mnb.fit(X_train_new, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0, 2.0, 5.0]}, scoring='f1')

In [39]:
grid_mnb.best_estimator_

MultinomialNB(alpha=0.1)

In [40]:
y_pred_mnb_ht = grid_mnb.best_estimator_.predict(X_test_new)

**check accuracy**

In [48]:
print("Accuracy: ", accuracy_score(y_test, y_pred_mnb_ht))
print("Precision: ", precision_score(y_test, y_pred_mnb_ht))
print("Recall: ", recall_score(y_test, y_pred_mnb_ht))
print("F1 Score: ", f1_score(y_test, y_pred_mnb_ht))

accuracy_dict['Multinomial Naive Bayes with hyper parameter tuning'] = accuracy_score(y_test, y_pred_mnb_ht)
precision_dict['Multinomial Naive Bayes with hyper parameter tuning'] = precision_score(y_test, y_pred_mnb_ht)
recall_dict['Multinomial Naive Bayes with hyper parameter tuning'] = recall_score(y_test, y_pred_mnb_ht)
f1_dict['Multinomial Naive Bayes with hyper parameter tuning'] = f1_score(y_test, y_pred_mnb_ht)

Accuracy:  0.9822119102861562
Precision:  0.9490445859872612
Recall:  0.9085365853658537
F1 Score:  0.9283489096573209


* considerable improvement achieved in _Recall_ and _F1 Score_ 
* slight improvement achieved in _Accuracy Score_
* but _Precision_ dropped 

# Comparing the models' performances

In [66]:
#print("Accuracy scores : ")
#accuracy_dict
accuracy_mod_dict = dict(sorted(accuracy_dict.items() ,key=lambda x: x[1], reverse=True))
print("Accuracy scores sorted in descending order : ")
accuracy_mod_dict

Accuracy scores sorted in descending order : 


{'Multinomial Naive Bayes with hyper parameter tuning': 0.9822119102861562,
 'Logistic Regression with hyper parameter tuning': 0.9783449342614076,
 'XGBoost Classifier': 0.9752513534416086,
 'XGBoost Classifier with hyper parameter tuning': 0.974477958236659,
 'Multinomial Naive Bayes': 0.974477958236659,
 'Logistic Regression': 0.9698375870069605}

In [65]:
#print("Precision scores : ")
#precision_dict
precision_mod_dict = dict(sorted(precision_dict.items() ,key=lambda x: x[1], reverse=True))
print("Precision scores sorted in descending order : ")
precision_mod_dict

Precision scores sorted in descending order : 


{'Multinomial Naive Bayes': 1.0,
 'Logistic Regression': 0.9921259842519685,
 'Multinomial Naive Bayes with hyper parameter tuning': 0.9490445859872612,
 'XGBoost Classifier': 0.9459459459459459,
 'XGBoost Classifier with hyper parameter tuning': 0.9395973154362416,
 'Logistic Regression with hyper parameter tuning': 0.9047619047619048}

In [64]:
#print("Recall scores : ")
#recall_dict
recall_mod_dict = dict(sorted(recall_dict.items() ,key=lambda x: x[1], reverse=True))
print("Recall scores sorted in descending order : ")
recall_mod_dict

Recall scores sorted in descending order : 


{'Logistic Regression with hyper parameter tuning': 0.926829268292683,
 'Multinomial Naive Bayes with hyper parameter tuning': 0.9085365853658537,
 'XGBoost Classifier': 0.8536585365853658,
 'XGBoost Classifier with hyper parameter tuning': 0.8536585365853658,
 'Multinomial Naive Bayes': 0.7987804878048781,
 'Logistic Regression': 0.7682926829268293}

In [63]:
# print("F1 scores : ")
# print(f1_dict)
f1_mod_dict = dict(sorted(f1_dict.items() ,key=lambda x: x[1], reverse=True))
print("F1 scores sorted in descending order : ")
f1_mod_dict

F1 scores sorted in descending order : 


{'Multinomial Naive Bayes with hyper parameter tuning': 0.9283489096573209,
 'Logistic Regression with hyper parameter tuning': 0.9156626506024096,
 'XGBoost Classifier': 0.8974358974358975,
 'XGBoost Classifier with hyper parameter tuning': 0.8945686900958466,
 'Multinomial Naive Bayes': 0.888135593220339,
 'Logistic Regression': 0.865979381443299}

# Conclusion : 

It can be concluded that the overall best performance is given by : **_Multinomial Naive Bayes with Hyper parameter tuning_**

# Testing on a single input 

In [71]:
message = input("Enter a message : ")
message_vector = tfidf.transform([message])

prediction = grid_mnb.best_estimator_.predict(message_vector)

if(prediction[0] == 1):
    print("The message is classified as : SPAM")
else:
    print("The message is classified as : HAM(not spam)")

The message is classified as : HAM(not spam)
